<a href="https://colab.research.google.com/github/eeeewyz/LLM/blob/main/hf_2_finetune_bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# =========================
#  加载预训练模型和分词器
# =========================

checkpoint = "bert-base-uncased"

# 加载 BERT 对应的 Tokenizer，将文本转换为 Token IDs
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# 加载 BERT 序列分类模型
# 在预训练 BERT 上添加一个分类头（Classification Head）
# 默认 num_labels=2，即二分类
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# （test）
# 计算总参数量和可训练参数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# 获取 BERT 的 Transformer 层数
num_layers = model.config.num_hidden_layers

print(f"模型总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")
print(f"Transformer 层数: {num_layers}")

模型总参数量: 109,483,778
可训练参数量: 109,483,778
Transformer 层数: 12


In [ ]:
# （test）
#一：简单测试仅仅训练三条数据
# 准备训练数据
# =========================

sequences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "This course is amazing!",
    "i hate this so much",
]

# 文本预处理
# padding=True：将 Batch 中的序列补齐到相同长度
# truncation=True：超过最大长度时截断
# return_tensors="pt"：返回 PyTorch Tensor
batch = tokenizer(
    sequences,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

#因为自己写的句子，所以没标签，下面自己设置一下
# 三个样本的标签设置为 1，1，0（例如表示正面情感）
batch["labels"] = torch.tensor([1, 1,0])
print(batch["labels"].shape)

torch.Size([3])


In [ ]:
# （test）
# 创建优化器
# =========================

# AdamW 优化器
# model.parameters()：获取模型所有可训练参数
# 优化器负责根据梯度更新模型参数
optimizer = AdamW(model.parameters())


# =========================
# Forward Pass（前向传播）
# =========================
# **batch：将字典展开为模型的关键字参数
# 包含 input_ids、attention_mask、labels 等
# 模型自动计算分类预测及 Cross-Entropy Loss
loss = model(**batch).loss

# =========================
# Backward Pass（反向传播）
# =========================
# Autograd 自动计算 Loss 对各个可训练参数的梯度
# 梯度保存在 parameter.grad 中
loss.backward()

# =========================
# 更新模型参数
# =========================

# AdamW 根据梯度更新模型参数
optimizer.step()

# 清除梯度，避免下一次训练时梯度累积
optimizer.zero_grad()

print("Training loss:", loss.item())

Training loss: 0.6877285838127136


In [ ]:
# 二：加载数据集训练
# 使用完整的命名空间路径避免 Hugging Face 库版本的单片段解析 Bug
from datasets import load_dataset

raw_datasets = load_dataset("nyu-mll/glue", "mrpc")

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mrpc/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  649kB            

mrpc/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 75.7kB            

mrpc/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  308kB            

mrpc/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

In [ ]:
# （test）
# 打印并查看数据集的具体结构与其中一条样本
print(raw_datasets)

# 这个数据集的任务是Sentence Pair Classification，即判断两句话含义是否相同
# label=1：两句话语义相同
# label=0：两句话语义不同
print("\n训练集第一条样本内容:")
print(raw_datasets["train"][0])

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

训练集第一条样本内容:
{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .', 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .', 'label': 1, 'idx': 0}


In [ ]:
# （test）
# 可以看出来raw_datasets 的label是整数int32
raw_datasets["train"].features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

In [ ]:
# 这一部分主要强调 Hugging Face Dataset 的高效预处理机制，
# 核心是 map()
# batched=True ：每次传入一个batch，并且tokenizer传入一批进去，而不是一句句处理
# Dynamic Padding（动态填充）：
# 把每个 Batch 内的所有样本 Padding 到该 Batch 中最长样本的长度，而不是整个 Dataset 的最大长度。
# 但是每个batch之间的长度可以不同，比如
# Batch 1: (3, 12)
# Batch 2: (3, 9)
# Batch 3: (3, 15)
# 这样可以加快预处理

In [ ]:
# 定义一个处理函数，同时接收 sentence1 和 sentence2
# Tokenizer 会将它们自动拼接，并生成 token_type_ids 用以区分两句话
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

# 使用 map 方法对整个 train_datasets和 进行批处理分词，它会自动处理数据类型并加速
tokenized_train_datasets = raw_datasets['train'].map(tokenize_function, batched=True)

# （test）
# 查看分词后的数据集结构
print(tokenized_train_datasets)

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 3668
})


In [ ]:
# （test）
#看下第一个sample的tokenid
#mask全是1可以看出没有padding
input_ids=tokenized_train_datasets[0]["input_ids"]
print(input_ids)

[101, 2572, 3217, 5831, 5496, 2010, 2567, 1010, 3183, 2002, 2170, 1000, 1996, 7409, 1000, 1010, 1997, 9969, 4487, 23809, 3436, 2010, 3350, 1012, 102, 7727, 2000, 2032, 2004, 2069, 1000, 1996, 7409, 1000, 1010, 2572, 3217, 5831, 5496, 2010, 2567, 1997, 9969, 4487, 23809, 3436, 2010, 3350, 1012, 102]


In [ ]:
# （test）
#反向把id化成tokens
tokenizer.convert_ids_to_tokens(input_ids)

['[CLS]',
 'am',
 '##ro',
 '##zi',
 'accused',
 'his',
 'brother',
 ',',
 'whom',
 'he',
 'called',
 '"',
 'the',
 'witness',
 '"',
 ',',
 'of',
 'deliberately',
 'di',
 '##stor',
 '##ting',
 'his',
 'evidence',
 '.',
 '[SEP]',
 'referring',
 'to',
 'him',
 'as',
 'only',
 '"',
 'the',
 'witness',
 '"',
 ',',
 'am',
 '##ro',
 '##zi',
 'accused',
 'his',
 'brother',
 'of',
 'deliberately',
 'di',
 '##stor',
 '##ting',
 'his',
 'evidence',
 '.',
 '[SEP]']

In [ ]:
#DataCollator 将多个样本整理成一个 Batch，
#并通过动态 Padding 使 Batch 内的序列长度一致，转换为模型可直接使用的 Tensor。
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
# （test）
# 1. 从已成功分词的 tokenized_train_datasets 中取出前 8 个样本
# 此时它已经是 Dataset 对象，直接切片即可，无需 ['train']
samples = tokenized_train_datasets[:8]


# 2. 删除模型不需要的字段
# samples.items()：遍历字典中的 key-value
# k：字段名称，v：对应的数据
# 只保留不在 ["idx", "sentence1", "sentence2"] 中的字段
# 最终保留 input_ids、attention_mask、token_type_ids、label
samples = {
    k: v
    for k, v in samples.items()
    if k not in ["idx", "sentence1", "sentence2"]
}


# 3. 查看 8 个样本各自的 Token 长度
# samples["input_ids"]：包含 8 个样本的 Token ID 列表
# for x in ...：依次取出每个样本的 Token IDs
# len(x)：计算当前样本的 Token 数量
[len(x) for x in samples["input_ids"]]

[50, 59, 47, 67, 59, 50, 62, 32]

In [ ]:
# （test）
# 1. 使用 Data Collator 对 8 个样本进行动态 Padding
# 将所有样本补齐到当前 Batch 中最长样本的长度
# 同时转换为 PyTorch Tensor
batch = data_collator(samples)

# 2. 查看 Batch 中每个字段的 Tensor Shape
# batch.items()：遍历字段名称 k 和对应的 Tensor v
# v.shape：查看该 Tensor 的形状
{k: v.shape for k, v in batch.items()}

{'input_ids': torch.Size([8, 67]),
 'token_type_ids': torch.Size([8, 67]),
 'attention_mask': torch.Size([8, 67]),
 'labels': torch.Size([8])}

In [ ]:
# （test）
#8 = Batch Size：一个 Batch 包含 8 对句子。
#67 = Sequence Length：这 8 对句子中，最长的一对经过 Tokenization 后有 67 个 Tokens。

In [ ]:
#三：准备微调
# TrainingArguments：Hugging Face 提供的训练参数配置类。
# training_args：创建出来的配置对象（Object）。
# "XXXXX"：文件夹路径，用来保存训练 Checkpoints 等输,可以自己起名字，这里面比如用test—trainer


# 它可能放的东西
# test-trainer/
# │
# ├── checkpoint-500/
# │   ├── model.safetensors      ← Model Parameters
# │   ├── optimizer.pt           ← Optimizer State
# │   ├── scheduler.pt           ← Scheduler State
# │   ├── trainer_state.json
# │   └── training_args.bin
# │
# ├── checkpoint-1000/
# │   ├── model.safetensors
# │   ├── optimizer.pt
# │   ├── scheduler.pt
# │   ├── trainer_state.json
# │   └── training_args.bin
# │
# └── ...


# TrainingArguments (Class)
#     │
#     └── training_args (Object)
#             │
#             ├── output_dir = "test-trainer"  ← 文件夹
#             ├── learning_rate
#             ├── batch_size
#             └── num_train_epochs

from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer")


In [ ]:
from transformers import Trainer

# 正式微调前重新加载预训练权重，避免前面的（test）手动训练影响模型参数
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_datasets,
    data_collator=data_collator,
    processing_class=tokenizer # 较新版本的 transformers 库中使用 processing_class 代替 tokenizer
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#因为没有设置epoch，所以默认为3
trainer.train()


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,0.533235
1000,0.309344


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1377, training_loss=0.36144051589910414, metrics={'train_runtime': 7870.4627, 'train_samples_per_second': 1.398, 'train_steps_per_second': 0.175, 'total_flos': 405114969714960.0, 'train_loss': 0.36144051589910414, 'epoch': 3.0})

In [ ]:
# predictions.predictions 得到的就是经过分类层（Classification Head），但尚未经过 Softmax 的原始分数，也就是 Logits。


# 1:BERT Encoder
# 提取句子的语义表示

# 2:Classification Head (Linear)
# Logits
# [1.2, 2.8]

# 3:Softmax
# Probabilities
# [0.168, 0.832]

# Argmax → Class 1

In [ ]:
#先把validation数据集tokenization
tokenized_validation_datasets = raw_datasets['validation'].map(tokenize_function, batched=True)


# 使用训练好的 Trainer 在验证集（Validation Set）上进行预测
# 返回 PredictionOutput，包含 predictions（预测结果）、label_ids（真实标签）等
predictions = trainer.predict(tokenized_validation_datasets)

# 查看预测结果和真实标签的形状
# predictions.predictions：模型输出的 Logits，形状为 (样本数, 类别数)
print(predictions.predictions.shape)


# predictions.label_ids：真实标签，形状为 (样本数,)
print(predictions.label_ids.shape)



Map:   0%|          | 0/408 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


(408, 2)
(408,)


In [ ]:
# 每一行都选择数值最大的那个位置（索引从 0 开始）。
#  axis=-1 是什么意思？
# axis 指定在哪个维度上寻找最大值。

# Logits → Argmax → Predicted Labels
# 样本	Class 0	Class 1	     预测类别
# 1	1.2	       2.8	             1
# 2	3.5	       0.7	             0
# 3	0.3	       1.9	             1

import numpy as np

# 将 Logits 转换为预测的类别
preds = np.argmax(predictions.predictions, axis=-1)


#preds 数组，就是模型对整个 Validation Dataset（验证集）中每一条样本预测出的最终类别标签。
preds

array([1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1,
       0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0,
       1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0,
       1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1,
       1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0,
       1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1,
       1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1,
       1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1,
       1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0,
       1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1,
       1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1,
       1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1,

In [ ]:
# evaluate.load("glue", "mrpc") 会自动加载 MRPC 任务对应的评估指标，
# 也就是 Accuracy 和 F1 Score。

In [ ]:
!pip install evaluate scikit-learn

import evaluate  # 导入 Hugging Face Evaluate 库，用于模型评估

# 加载 GLUE 基准测试中 MRPC 任务的评估指标（Accuracy 和 F1 Score）
metric = evaluate.load("glue", "mrpc")

# 将模型预测标签（preds）与真实标签（label_ids）进行比较
# predictions：模型预测的类别标签（0 或 1）
# references：验证集的真实标签（Ground Truth）
# 返回 Accuracy（准确率）和 F1 Score
metric.compute(predictions=preds, references=predictions.label_ids)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


{'accuracy': 0.8627450980392157, 'f1': 0.9041095890410958}